In [1]:
# Kill all processess on GPU
!fuser -v /dev/nvidia* -k

In [2]:
# Check GPU status
!nvidia-smi

Sat Feb 28 19:45:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   65C    P0             28W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
%%capture
import os
os.environ["UNSLOTH_VLLM_STANDBY"] = "1" # [NEW] Extra 30% context lengths!
if "COLAB_" not in "".join(os.environ.keys()):
    # If you're not in Colab, just use pip install or uv pip install
    %pip install unsloth vllm
else:
    %pip install --upgrade -qqq uv
    !uv pip install vllm==0.11.2 unsloth-zoo unsloth
    !uv pip install transformers==4.56.2
    !uv pip install --no-deps trl==0.22.2

# Configuration

In [13]:
# Model configuration
# MODEL_ID = 'unsloth/Qwen2.5-VL-3B-Instruct'
MODEL_ID = 'unsloth/Qwen2.5-VL-7B-Instruct-bnb-4bit'
LOAD_IN_4BIT = False # False for LoRA 16bit
MAX_SEQ_LENGTH = 16384 # Must be this long for VLMs

# Data configuration
DATA_ID = 'jxie/coco_captions'
DATA_SPLIT = 'train'
DATA_SIZE = 1000
BATCH_SIZE = 10
SAVE_DATA_ID = 'alxxtexxr/BIVR-Data'

# Model

In [5]:
from unsloth import FastVisionModel
import torch

model, tokenizer = FastVisionModel.from_pretrained(
    model_name = MODEL_ID,
    load_in_4bit = LOAD_IN_4BIT,
    max_seq_length = MAX_SEQ_LENGTH,
    # fast_inference = True, # Enable vLLM fast inference
    # fast_inference = False, # Disable to fix the vLLM bug on T4
    # gpu_memory_utilization = 0.8, # Reduce if out of memory
    # torch_dtype = torch.float32,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
ERROR 02-28 19:46:12 [fa_utils.py:64] Cannot use FA version 2 is not supported due to FA2 is only supported on devices with compute capability >= 8
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.2.1: Fast Qwen2_5_Vl patching. Transformers: 4.56.2. vLLM: 0.11.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [6]:
from datasets import load_dataset, Dataset

def load_hf_dataset(
    data_id,
    split='train',
    train_size = 1000,
    test_size = 0,
):
    # Use streaming
    dataset_stream = load_dataset(data_id, split=split, streaming=True)

    # Manually take train_size + test_size samples
    total_size = train_size + test_size
    sliced_data = []
    
    dataset_size = 0
    unique_cocoids = set()
    
    for example in dataset_stream:
        if dataset_size >= total_size:
            break
        
        # Ensure unique cocoids if cocoid exists
        cocoid = example.get('cocoid', None)
        if cocoid is not None:
            if cocoid in unique_cocoids:
                continue
            unique_cocoids.add(cocoid)
        
        sliced_data.append(example)
        dataset_size += 1

    # Convert to regular in-memory dataset
    dataset = Dataset.from_list(sliced_data)
    
    return dataset


dataset = load_hf_dataset(DATA_ID, split=DATA_SPLIT, train_size=DATA_SIZE)

Resolving data files:   0%|          | 0/182 [00:00<?, ?it/s]

In [7]:
import requests
from PIL import Image
from io import BytesIO

def process_image_with_url(example):
    url = example['url']

    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()

        image = Image.open(BytesIO(response.content))
        image.load()

        # Convert to RGB
        if image.mode != "RGB":
            image = image.convert("RGB")

        # Resize
        image = image.resize((512, 512), Image.LANCZOS)

    except Exception as e:
        print(f"Failed to process {url}: {e}")
        image = None

    example["decoded_image"] = image
    return example

def process_image(example):
    image_col = 'decoded_image' if 'decoded_image' in example else 'image'
    image = example[image_col]
    image = image.resize((512, 512), Image.LANCZOS)
    if image.mode != 'RGB':
        image = image.convert('RGB')
    example['decoded_image'] = image
    return example

if 'url' in dataset.features:
    # Load and process images from URLs
    dataset = dataset.map(
        process_image_with_url, 
        # num_proc=4, # Somehow multiprocessing causes errors on T4, 
                      # maybe because of too many threads trying to download images at once. 
                      # You can try num_proc=2 or 4 if you have a better GPU
    )
else:
    # Resize to (512, 512) and convert to RGB
    dataset = dataset.map(
        process_image, 
        # num_proc=4,
    )

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [8]:
# Remove the original 'image' column
train_dataset = dataset.remove_columns('image')

# Rename 'decoded_image' to 'image'
train_dataset = train_dataset.rename_column('decoded_image', 'image')

In [9]:
# # Remove bad data point with problematic vision embedding
# bad_pid = '36'
# train_dataset = train_dataset.filter(lambda x: x['pid'] != bad_pid)

In [10]:
# from vllm import SamplingParams

# sampling_params = SamplingParams(
#     temperature = 1.0,
#     top_k = 50,
#     max_tokens = 1024,
# )

# outputs = model.fast_generate(
#     {
#         "prompt": train_dataset[100]["prompt"],
#         "multi_modal_data": {"image": train_dataset[100]["image"]}
#     },
#     sampling_params,
# )
# print(outputs[0].outputs[0].text)

In [11]:
from transformers import AutoProcessor

processor = AutoProcessor.from_pretrained(MODEL_ID)

In [ ]:
import math
import os
from pathlib import Path
import numpy as np
from huggingface_hub import upload_file
from sklearn.preprocessing import StandardScaler

train_size = len(train_dataset)
save_dir = Path(f'model_{MODEL_ID.replace("/", "_")}.data_{DATA_ID.replace("/", "_")}_{DATA_SPLIT}')
os.makedirs(save_dir, exist_ok=True)

scaler = StandardScaler()

for i in range(math.ceil(train_size / BATCH_SIZE)):
    start_idx = i * BATCH_SIZE
    end_idx = min(start_idx + BATCH_SIZE, train_size)
    range_tag = f'{start_idx}-{end_idx-1}'
    
    print("="*128)
    print("Range:", range_tag)
    print("="*128)
    
    inputs = processor.image_processor(images=list(train_dataset['image'][start_idx:end_idx]), return_tensors='pt')
    pixel_values = inputs['pixel_values'].to(model.device)
    image_grid_thw = inputs['image_grid_thw'].to(model.device)
    
    with torch.no_grad():
        visual_embs = model.visual(pixel_values, image_grid_thw)

    print("Pixel values shape:", pixel_values.shape)
    print("Image grid shape:", image_grid_thw.shape)
    print("Visual embeddings shape:", visual_embs.shape)
    print()
    
    # batch = visual_embs.float() # Convert to float32 for statistics calculation -> (N, 2048)

    # # Statistics for Standardization
    # batch_n = batch.shape[0]
    # batch_sum = batch.sum(dim=0)
    # batch_sum_sq = (batch**2).sum(dim=0)

    # # Statistics for Min-Max Normalization
    # batch_min = batch.min(dim=0).values
    # batch_max = batch.max(dim=0).values

    # print("Batch size:", batch_n)
    # print("Batch sum average:", batch_sum.mean())
    # print("Batch sum squared average:", batch_sum_sq.mean())
    # print("Batch min average:", batch_min.mean())
    # print("Batch max average:", batch_max.mean())
    # print()

    # Partial fit the scaler with current batch
    scaler.partial_fit(visual_embs.detach().cpu().numpy())

    # Save vision data and statistics
    range_tag = f'{start_idx}-{end_idx-1}' # TODO: Implement zfill for better sorting
    vision_data_path = save_dir / f'{range_tag}.vision_data.npz'
    stats_path = save_dir / f'{range_tag}.stats.npz'
    
    np.savez(
        vision_data_path,
        pixel_values=pixel_values.cpu().numpy(),
        image_grid_thw=image_grid_thw.cpu().numpy(),
        visual_embs=visual_embs.cpu().numpy()
    )
    # np.savez(
    #     stats_path,
    #     batch_n=batch_n,
    #     batch_sum=batch_sum.cpu().numpy(),
    #     batch_sum_sq=batch_sum_sq.cpu().numpy(),
    #     batch_min=batch_min.cpu().numpy(),
    #     batch_max=batch_max.cpu().numpy()
    # )

    print("Vision data saved to:", vision_data_path)
    # print("Statistics saved to:", stats_path)

    upload_file(
        path_or_fileobj=str(vision_data_path),
        path_in_repo=str(vision_data_path),
        repo_id=SAVE_DATA_ID,
        repo_type='dataset',
    )
    # upload_file(
    #     path_or_fileobj=str(stats_path),
    #     path_in_repo=str(stats_path),
    #     repo_id=SAVE_DATA_ID,
    #     repo_type='dataset',
    # )

Range: 0-9
Pixel values shape: torch.Size([12960, 1176])
Image grid shape: torch.Size([10, 3])
Visual embeddings shape: torch.Size([3240, 3584])

Vision data saved to: model_unsloth_Qwen2.5-VL-7B-Instruct-bnb-4bit.data_jxie_coco_captions_train/0-9.vision_data.npz


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...train/0-9.vision_data.npz:  57%|#####6    | 47.8MB / 84.2MB            

Range: 10-19
Pixel values shape: torch.Size([12960, 1176])
Image grid shape: torch.Size([10, 3])
Visual embeddings shape: torch.Size([3240, 3584])

Vision data saved to: model_unsloth_Qwen2.5-VL-7B-Instruct-bnb-4bit.data_jxie_coco_captions_train/10-19.vision_data.npz


...

  ...n/980-989.vision_data.npz:  47%|####7     | 39.9MB / 84.2MB            

Range: 990-999
Pixel values shape: torch.Size([12960, 1176])
Image grid shape: torch.Size([10, 3])
Visual embeddings shape: torch.Size([3240, 3584])

Vision data saved to: model_unsloth_Qwen2.5-VL-7B-Instruct-bnb-4bit.data_jxie_coco_captions_train/990-999.vision_data.npz


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...n/990-999.vision_data.npz:  46%|####6     | 39.0MB / 84.2MB            

In [ ]:
import joblib

# Save the scaler
scaler_path = save_dir / 'scaler.pkl'
joblib.dump(scaler, scaler_path)

print("Scaler saved to:", scaler_path)

upload_file(
    path_or_fileobj=str(scaler_path),
    path_in_repo=str(scaler_path),
    repo_id='alxxtexxr/BIVR-Data',
    repo_type='dataset',
)

Scaler saved to: model_unsloth_Qwen2.5-VL-7B-Instruct-bnb-4bit.data_jxie_coco_captions_train/scaler.pkl


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...captions_train/scaler.pkl: 100%|##########| 86.6kB / 86.6kB            

CommitInfo(commit_url='https://huggingface.co/datasets/alxxtexxr/BIVR-Data/commit/313e7e3368ea3f20a4d842024a38e14ab1422b22', commit_message='Upload model_unsloth_Qwen2.5-VL-7B-Instruct-bnb-4bit.data_jxie_coco_captions_train/scaler.pkl with huggingface_hub', commit_description='', oid='313e7e3368ea3f20a4d842024a38e14ab1422b22', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/alxxtexxr/BIVR-Data', endpoint='https://huggingface.co', repo_type='dataset', repo_id='alxxtexxr/BIVR-Data'), pr_revision=None, pr_num=None)

: 